# LeetCode 1370 — Increasing Decreasing String bằng Python

**Link đề:** https://leetcode.com/problems/increasing-decreasing-string/
**Độ khó:** Easy · **Chủ đề:** Hash Table, String, Counting

```python
class Solution:
    def sortString(self, s: str) -> str:
        ...
```

## Mục tiêu học
1. Đọc hiểu một thuật toán được mô tả bằng **các bước** và chuyển thành code **mô phỏng** (simulation).
2. Nhận ra rằng chỉ **số lần xuất hiện** của mỗi chữ cái là quan trọng → dùng **mảng đếm 26 ô**.
3. Dùng thành thạo `ord`, `chr`, `range` ngược, `''.join`, `collections.Counter`.
4. So sánh 3 lời giải: mô phỏng trên list đã sắp → mảng đếm 26 ô (chính) → `Counter` + danh sách khóa.
5. Trace từng vòng, chứng minh, phân tích Time/Space, test đối chiếu và tự giải biến thể.

**Cách học:** đọc markdown → đoán output → chạy → so sánh. Cell TODO không tự chạy.

## 1. Đề bài

Cho chuỗi `s`. Sắp xếp lại theo thuật toán:
1. Lấy **ký tự nhỏ nhất** ra khỏi `s`, nối vào kết quả.
2. Lấy ký tự nhỏ nhất **lớn hơn** ký tự vừa nối, nối vào kết quả.
3. Lặp bước 2 tới khi không lấy được nữa.
4. Lấy **ký tự lớn nhất** ra khỏi `s`, nối vào kết quả.
5. Lấy ký tự lớn nhất **nhỏ hơn** ký tự vừa nối, nối vào kết quả.
6. Lặp bước 5 tới khi không lấy được nữa.
7. Lặp lại 1–6 cho tới khi lấy hết ký tự.

Nếu ký tự nhỏ/lớn nhất xuất hiện nhiều lần, lấy lần nào cũng được (kết quả như nhau).

| s | Output |
|---|---|
| `"aaaabbbbcccc"` | `"abccbaabccba"` |
| `"rat"` | `"art"` |

**Constraints:** `1 <= s.length <= 500`, `s` chỉ gồm chữ thường tiếng Anh.

Diễn giải bằng lời: mỗi vòng ta đi **lên** qua các chữ cái còn lại (mỗi chữ một lần, tăng nghiêm ngặt), rồi đi **xuống** (giảm nghiêm ngặt), cho tới khi hết.

### 1.1. Làm tay ví dụ 1

`s = "aaaabbbbcccc"` → còn `a:4, b:4, c:4`.

| Vòng | Pha | Lấy | Còn lại | Kết quả |
|---|---|---|---|---|
| 1 | lên | a, b, c | a:3 b:3 c:3 | `abc` |
| 1 | xuống | c, b, a | a:2 b:2 c:2 | `abccba` |
| 2 | lên | a, b, c | a:1 b:1 c:1 | `abccbaabc` |
| 2 | xuống | c, b, a | hết | `abccbaabccba` |

Ví dụ `"rat"`: pha lên lấy `a, r, t` → `"art"`, hết luôn.

## 2. Nền tảng Python cho bài

### 2.1. So sánh ký tự và `ord` / `chr`

Python so sánh chữ cái theo **mã Unicode**: `'a' < 'b' < ... < 'z'`. `ord(ch)` trả mã số, `chr(k)` làm ngược lại. Với chữ thường: `ord(ch) - ord('a')` cho chỉ số `0..25`.

In [1]:
print('a' < 'b', 'z' > 'r')
print(ord('a'), ord('b'), ord('z'))
for ch in "art":
    print(ch, "→ chỉ số", ord(ch) - ord('a'))
print(chr(ord('a') + 17))     # chỉ số 17 → 'r'

True True
97 98 122
a → chỉ số 0
r → chỉ số 17
t → chỉ số 19
r


### 2.2. `range` tiến và lùi

- `range(26)`: `0, 1, ..., 25` (đi lên).
- `range(25, -1, -1)`: `25, 24, ..., 0` (đi xuống, dừng **trước** `-1`).
- `reversed(range(26))` cũng cho dãy đi xuống.

In [2]:
print(list(range(5)))
print(list(range(4, -1, -1)))
print(list(reversed(range(5))))

[0, 1, 2, 3, 4]
[4, 3, 2, 1, 0]
[4, 3, 2, 1, 0]


### 2.3. Nối chuỗi hiệu quả: `list` + `''.join`

Chuỗi trong Python **bất biến** (immutable): `res += ch` tạo chuỗi mới mỗi lần. Cách chuẩn: gom ký tự vào **list** rồi `''.join(list)` một lần ở cuối.

In [3]:
parts = []
for ch in "abc":
    parts.append(ch)
parts.append("x")
print(parts, "→", "".join(parts))
print("-".join(["a", "b", "c"]))   # chuỗi trước .join là "keo" nối

['a', 'b', 'c', 'x'] → abcx
a-b-c


### 2.4. Mảng đếm 26 ô

Mảng `count` dài 26, `count[k]` = số lần chữ cái thứ `k` xuất hiện. Đây là "dict" đơn giản nhất khi key là chữ thường.

In [4]:
s = "aaaabbbbcccc"
count = [0] * 26
for ch in s:
    count[ord(ch) - ord('a')] += 1
print(count[:5], "...")
print({chr(k + ord('a')): count[k] for k in range(26) if count[k]})

[4, 4, 4, 0, 0] ...
{'a': 4, 'b': 4, 'c': 4}


Dòng cuối là **dict comprehension**: `{key: value for ... if ...}` — ở đây chỉ in các chữ cái có mặt.

## 3. Cách 1 — Baseline: mô phỏng đúng từng bước trên list

Làm **y như đề**: giữ các ký tự còn lại trong list đã sắp `rest`.
- Pha lên: duyệt `rest` từ trái sang phải; lấy ký tự nếu nó **lớn hơn** ký tự vừa lấy (hoặc là ký tự đầu tiên của pha).
- Pha xuống: duyệt từ phải sang trái; lấy ký tự nếu nó **nhỏ hơn** ký tự vừa lấy.
- Ký tự đã lấy thì bỏ khỏi `rest`.

In [5]:
def sort_string_simulate(s):
    rest = sorted(s)               # list ký tự tăng dần
    res = []
    while rest:
        # Pha lên
        last = None
        keep = []
        for ch in rest:
            if last is None or ch > last:
                res.append(ch)
                last = ch
            else:
                keep.append(ch)
        rest = keep
        # Pha xuống
        last = None
        keep = []
        for ch in reversed(rest):
            if last is None or ch < last:
                res.append(ch)
                last = ch
            else:
                keep.append(ch)
        rest = keep[::-1]          # keep đang ngược, lật lại cho tăng dần
    return "".join(res)

print(sort_string_simulate("aaaabbbbcccc"))   # abccbaabccba
print(sort_string_simulate("rat"))            # art

abccbaabccba
art


Giải thích chi tiết:
- `sorted(s)` với chuỗi trả **list ký tự** đã sắp: `sorted("rat") == ['a', 'r', 't']`.
- `while rest:` — list rỗng được coi là `False` → dừng khi lấy hết.
- `last is None` — `None` là giá trị "chưa có gì"; dùng `is` để so sánh với `None`.
- Vì `rest` đã sắp, trong pha lên, ký tự trùng với `last` sẽ bị giữ lại; ký tự lớn hơn được lấy — đúng "nhỏ nhất lớn hơn ký tự vừa nối".

**Chi phí:** mỗi vòng ngoài duyệt cả `rest` (≤ n) hai lần. Số vòng ngoài ≈ (số lần xuất hiện lớn nhất của một chữ) / 2. Trường hợp xấu `s = "aaaa...a"` (n chữ `a`): mỗi pha chỉ lấy được 1 ký tự → khoảng `n/2` vòng × `O(n)` = **`O(n²)`**.

In [6]:
import time
s_bad = "a" * 500
t0 = time.perf_counter()
sort_string_simulate(s_bad)
print(f"n=500 toàn 'a': {time.perf_counter() - t0:.4f} giây (vẫn nhanh, nhưng tăng bậc hai)")

n=500 toàn 'a': 0.0040 giây (vẫn nhanh, nhưng tăng bậc hai)


## 4. Insight — chỉ cần biết **còn bao nhiêu** mỗi chữ

Trong mỗi pha:
- Pha lên lấy **đúng một** bản của **mỗi** chữ cái còn lại, theo thứ tự `a → z`.
- Pha xuống lấy **đúng một** bản của mỗi chữ cái còn lại, theo thứ tự `z → a`.

Vì sao? Pha lên bắt đầu từ chữ nhỏ nhất còn lại, rồi mỗi bước lấy chữ nhỏ nhất **lớn hơn hẳn** — tức chữ **kế tiếp** (khác) trong bảng chữ cái còn hàng. Không chữ nào bị lấy hai lần (vì phải tăng nghiêm ngặt) và không chữ nào còn hàng bị bỏ qua.

Vậy ta không cần list ký tự hay xoá phần tử — chỉ cần **mảng đếm 26 ô** và hai vòng `for` theo bảng chữ cái:

```text
while còn ký tự:
    for k = 0..25:   nếu count[k] > 0: lấy chữ k, count[k] -= 1
    for k = 25..0:   nếu count[k] > 0: lấy chữ k, count[k] -= 1
```

## 5. Cách 2 (chính) — Mảng đếm 26 ô

Mổ xẻ từng đoạn:
1. Đếm: `count[ord(ch) - 97] += 1` (`97 == ord('a')`).
2. Lặp tới khi `len(res) == len(s)` (đã lấy đủ).
3. Pha lên: `for k in range(26)`; pha xuống: `for k in range(25, -1, -1)`.
4. Ghép kết quả: `''.join(res)`.

In [7]:
def sort_string_count(s):
    count = [0] * 26
    for ch in s:
        count[ord(ch) - ord('a')] += 1

    res = []
    while len(res) < len(s):
        for k in range(26):                 # pha lên a → z
            if count[k] > 0:
                res.append(chr(k + ord('a')))
                count[k] -= 1
        for k in range(25, -1, -1):         # pha xuống z → a
            if count[k] > 0:
                res.append(chr(k + ord('a')))
                count[k] -= 1
    return "".join(res)

print(sort_string_count("aaaabbbbcccc"))   # abccbaabccba
print(sort_string_count("rat"))            # art

abccbaabccba
art


## 6. TRACE từng pha

Ví dụ `s = "leetcode"`: đếm `c:1, d:1, e:3, l:1, o:1, t:1`.

| Vòng | Pha | Lấy | count sau pha | res |
|---|---|---|---|---|
| 1 | lên | c d e l o t | e:2 | `cdelot` |
| 1 | xuống | e | e:1 | `cdelote` |
| 2 | lên | e | (hết) | `cdelotee` |
| 2 | xuống | — | | `cdelotee` |

Chú ý pha xuống của vòng 2 **không lấy gì** — vòng `while` kiểm tra điều kiện rồi dừng.

In [8]:
def show_count(count):
    return "{" + ", ".join(f"{chr(k + 97)}:{count[k]}" for k in range(26) if count[k]) + "}"

def trace_sort_string(s):
    count = [0] * 26
    for ch in s:
        count[ord(ch) - 97] += 1
    print("Đếm ban đầu:", show_count(count))
    res = []
    round_no = 1
    while len(res) < len(s):
        for phase, ks in (("lên   ", range(26)), ("xuống ", range(25, -1, -1))):
            taken = []
            for k in ks:
                if count[k] > 0:
                    taken.append(chr(k + 97))
                    count[k] -= 1
            res.extend(taken)
            print(f"Vòng {round_no} pha {phase}: lấy {''.join(taken) or '—':<8} | còn {show_count(count):<20} | res = {''.join(res)}")
        round_no += 1
    return "".join(res)

trace_sort_string("leetcode")
print()
trace_sort_string("aaaabbbbcccc")

Đếm ban đầu: {c:1, d:1, e:3, l:1, o:1, t:1}
Vòng 1 pha lên   : lấy cdelot   | còn {e:2}                | res = cdelot
Vòng 1 pha xuống : lấy e        | còn {e:1}                | res = cdelote
Vòng 2 pha lên   : lấy e        | còn {}                   | res = cdelotee
Vòng 2 pha xuống : lấy —        | còn {}                   | res = cdelotee

Đếm ban đầu: {a:4, b:4, c:4}
Vòng 1 pha lên   : lấy abc      | còn {a:3, b:3, c:3}      | res = abc
Vòng 1 pha xuống : lấy cba      | còn {a:2, b:2, c:2}      | res = abccba
Vòng 2 pha lên   : lấy abc      | còn {a:1, b:1, c:1}      | res = abccbaabc
Vòng 2 pha xuống : lấy cba      | còn {}                   | res = abccbaabccba


'abccbaabccba'

Trong trace có cú pháp `''.join(taken) or '—'`: nếu chuỗi rỗng (coi là `False`) thì `or` trả về vế phải `'—'` — mẹo in "không có gì".

Cũng có `for phase, ks in ((...), (...))`: duyệt một tuple các cặp, mỗi cặp được **unpack** thành `phase` và `ks`.

## 7. Cách 3 — `Counter` + danh sách chữ cái đã sắp

`collections.Counter(s)` đếm tần suất thành dict `{ký tự: số lần}`. Ta lấy danh sách **chữ cái phân biệt đã sắp** `keys = sorted(cnt)`, rồi mỗi pha duyệt `keys` (lên) và `reversed(keys)` (xuống). Ưu điểm: không cần `ord/chr`, áp dụng được cho **bất kỳ** bảng ký tự nào (chữ hoa, số...).

In [9]:
from collections import Counter

cnt = Counter("leetcode")
print(cnt)
print(sorted(cnt))           # duyệt dict = duyệt key
print(cnt["e"], cnt["z"])    # key không có → 0 (Counter không báo lỗi)

Counter({'e': 3, 'l': 1, 't': 1, 'c': 1, 'o': 1, 'd': 1})
['c', 'd', 'e', 'l', 'o', 't']
3 0


In [10]:
def sort_string_counter(s):
    cnt = Counter(s)
    keys = sorted(cnt)
    res = []
    while len(res) < len(s):
        for ch in keys + keys[::-1]:        # lên rồi xuống trong một vòng
            if cnt[ch] > 0:
                res.append(ch)
                cnt[ch] -= 1
    return "".join(res)

print(sort_string_counter("aaaabbbbcccc"), sort_string_counter("rat"), sort_string_counter("leetcode"))

abccbaabccba art cdelotee


`keys + keys[::-1]` nối danh sách tăng dần với chính nó đảo ngược: `['a','b','c'] + ['c','b','a']`. Một vòng `for` thực hiện cả hai pha.

Cẩn thận: ở chỗ "giao" giữa hai pha, chữ lớn nhất có thể được lấy **hai lần liên tiếp** (`...c` rồi `c...`) — đúng theo đề (ví dụ `"abccba"`), vì pha xuống bắt đầu lại từ "lớn nhất còn lại".

## 8. Code nộp LeetCode

In [11]:
class Solution:
    def sortString(self, s: str) -> str:
        count = [0] * 26
        for ch in s:
            count[ord(ch) - 97] += 1
        res = []
        order = list(range(26)) + list(range(25, -1, -1))
        while len(res) < len(s):
            for k in order:
                if count[k]:
                    res.append(chr(k + 97))
                    count[k] -= 1
        return "".join(res)

sol = Solution()
print(sol.sortString("aaaabbbbcccc"))   # abccbaabccba
print(sol.sortString("rat"))            # art

abccbaabccba
art


`if count[k]:` — số `0` được coi là `False`, số khác 0 là `True`; tương đương `if count[k] > 0` (vì count không âm).

## 9. Vì sao đúng?

**Mệnh đề:** pha lên của đề (bước 1–3) lấy đúng một bản của mỗi chữ cái đang còn, theo thứ tự tăng dần.

*Chứng minh:* gọi các chữ còn hàng là `c1 < c2 < ... < cr`. Bước 1 lấy `c1` (nhỏ nhất). Giả sử vừa lấy `ci`; ký tự nhỏ nhất **lớn hơn hẳn** `ci` trong phần còn lại là `c(i+1)` (các bản còn lại của `ci` bằng `ci`, không lớn hơn; các chữ `< ci` cũng không). Lấy tới `cr` thì không còn ký tự nào lớn hơn → dừng. Đối xứng cho pha xuống.

Vòng `for k in range(26)` với điều kiện `count[k] > 0` lấy **đúng** tập `{c1, ..., cr}` theo thứ tự tăng → khớp đề. Mỗi lần lấy giảm `count` → tổng số lần lấy bằng `len(s)` → vòng `while` dừng.

## 10. Phân tích Time / Space

Gọi `n = len(s)`, `A = 26` (kích thước bảng chữ cái), `f` = tần suất lớn nhất của một chữ.

| Cách | Thời gian | Bộ nhớ phụ |
|---|---|---|
| 1. Mô phỏng trên list | `O(n · f)` ≈ `O(n²)` worst | `O(n)` |
| 2. Mảng đếm 26 | `O(n + A · f)` ≤ `O(A · n)` | `O(A + n)` (kết quả) |
| 3. Counter + keys | `O(n + r · f)`, `r` = số chữ phân biệt | `O(r + n)` |

Với Cách 2: mỗi vòng `while` tốn `2A = 52` bước và lấy ít nhất 1 ký tự; số vòng ≈ `f/2` nên tổng `O(n + A·f)`. Vì `A` là hằng số, thường viết `O(n)`.

In [12]:
def rounds_needed(s):
    return max(Counter(s).values()) // 2 + max(Counter(s).values()) % 2

for t in ["rat", "aaaabbbbcccc", "a" * 500, "abcdefghijklmnopqrstuvwxyz" * 19]:
    print(f"len={len(t):>3}, tần suất max={max(Counter(t).values()):>3}, số vòng while ≈ {rounds_needed(t)}")

len=  3, tần suất max=  1, số vòng while ≈ 1
len= 12, tần suất max=  4, số vòng while ≈ 2
len=500, tần suất max=500, số vòng while ≈ 250
len=494, tần suất max= 19, số vòng while ≈ 10


## 11. Bộ test: mẫu đề, biên, phản ví dụ, đối chiếu

In [13]:
import random

def check(name, actual, expected):
    assert actual == expected, (name, actual, expected)
    print(f"PASS {name}: {actual!r}")

sol = Solution()
check("đề 1", sol.sortString("aaaabbbbcccc"), "abccbaabccba")
check("đề 2", sol.sortString("rat"), "art")
check("một ký tự", sol.sortString("z"), "z")
check("toàn một chữ", sol.sortString("ggggg"), "ggggg")
check("leetcode", sol.sortString("leetcode"), "cdelotee")
check("spo", sol.sortString("spo"), "ops")
check("ggag", sol.sortString("ggag"), "aggg")
check("hai chữ lệch", sol.sortString("aab"), "aba")

PASS đề 1: 'abccbaabccba'
PASS đề 2: 'art'
PASS một ký tự: 'z'
PASS toàn một chữ: 'ggggg'
PASS leetcode: 'cdelotee'
PASS spo: 'ops'
PASS ggag: 'aggg'
PASS hai chữ lệch: 'aba'


In [14]:
methods = [sort_string_simulate, sort_string_count, sort_string_counter, Solution().sortString]
rng = random.Random(1370)
for _ in range(400):
    alphabet = "abcdefghijklmnopqrstuvwxyz"[: rng.randint(1, 26)]
    t = "".join(rng.choice(alphabet) for _ in range(rng.randint(1, 60)))
    expected = sort_string_simulate(t)
    for f in methods:
        assert f(t) == expected, (f, t)
    assert sorted(expected) == sorted(t)        # là hoán vị của s
print(f"400 chuỗi ngẫu nhiên × {len(methods)} cách: khớp nhau và là hoán vị — PASS")

400 chuỗi ngẫu nhiên × 4 cách: khớp nhau và là hoán vị — PASS


`f"{x!r}"` in giá trị dạng `repr` (chuỗi có dấu nháy) — dễ thấy chuỗi rỗng hay khoảng trắng.

Kiểm tra `sorted(expected) == sorted(t)` là **tính chất** của đáp án: kết quả phải là một **hoán vị** của `s` (cùng ký tự, cùng số lượng).

## 12. Lỗi người mới hay gặp

### Lỗi 1: Lấy hết các bản của một chữ trong một pha
Đề yêu cầu tăng **nghiêm ngặt** (`>`), nên mỗi chữ chỉ được lấy **một lần** mỗi pha.

In [15]:
def wrong_take_all(s):
    count = [0] * 26
    for ch in s:
        count[ord(ch) - 97] += 1
    res = []
    for k in range(26):                         # lấy hết luôn
        res.append(chr(k + 97) * count[k])
    return "".join(res)

print("Sai :", wrong_take_all("aaaabbbbcccc"), "| Đúng:", sort_string_count("aaaabbbbcccc"))

Sai : aaaabbbbcccc | Đúng: abccbaabccba


### Lỗi 2: Quên pha xuống
Chỉ lặp pha lên sẽ ra `"abcabcabcabc"` thay vì `"abccbaabccba"`.

In [16]:
def wrong_only_up(s):
    count = [0] * 26
    for ch in s:
        count[ord(ch) - 97] += 1
    res = []
    while len(res) < len(s):
        for k in range(26):
            if count[k]:
                res.append(chr(k + 97))
                count[k] -= 1
    return "".join(res)

print("Sai :", wrong_only_up("aaaabbbbcccc"), "| Đúng:", sort_string_count("aaaabbbbcccc"))

Sai : abcabcabcabc | Đúng: abccbaabccba


### Lỗi 3: Pha xuống dùng `range(25, 0, -1)`
`range(25, 0, -1)` dừng **trước** 0 → bỏ sót chữ `'a'` trong pha xuống. Phải là `range(25, -1, -1)`.

In [17]:
print("range(25, 0, -1) kết thúc ở:", list(range(25, 0, -1))[-1])
print("range(25, -1, -1) kết thúc ở:", list(range(25, -1, -1))[-1])

def wrong_range(s):
    count = [0] * 26
    for ch in s:
        count[ord(ch) - 97] += 1
    res = []
    guard = 0
    while len(res) < len(s) and guard < 1000:   # guard tránh lặp vô hạn khi minh hoạ
        guard += 1
        for k in range(26):
            if count[k]:
                res.append(chr(k + 97)); count[k] -= 1
        for k in range(25, 0, -1):              # thiếu k = 0
            if count[k]:
                res.append(chr(k + 97)); count[k] -= 1
    return "".join(res)

print("Sai :", wrong_range("aaaabbbbcccc"), "| Đúng:", sort_string_count("aaaabbbbcccc"))

range(25, 0, -1) kết thúc ở: 1
range(25, -1, -1) kết thúc ở: 0
Sai : abccbabccbaa | Đúng: abccbaabccba


Ở đây lỗi vẫn ra chuỗi cùng độ dài nhưng **thứ tự sai** (chữ `a` chỉ được lấy ở pha lên). Dấu `;` cho phép viết hai lệnh trên một dòng — chỉ dùng cho code minh hoạ ngắn.

### Lỗi 4: Nối chuỗi bằng `+=` trong vòng lặp lớn
Với `n = 500` không sao, nhưng thói quen `res += ch` với chuỗi rất dài có thể thành `O(n²)`. Hãy dùng list + `join`.

### Mẹo nhận dạng pattern

- Đề mô tả **quy trình lấy ra/đặt vào** theo thứ tự giá trị, và chỉ quan tâm **giá trị** (không quan tâm vị trí gốc) → **đếm tần suất**, bỏ cấu trúc vị trí.
- Bảng ký tự nhỏ (26 chữ) → mảng đếm `[0] * 26` với `ord(ch) - ord('a')`.
- "Lấy nhỏ nhất lớn hơn x" trên tập có đếm → duyệt bảng chữ cái theo thứ tự, bỏ qua ô `0`.
- Bài anh em: *Sort Characters By Frequency* (451), *Reorganize String* (767), *Custom Sort String* (791), *Rearrange Characters to Make Target String* (2287).

## 13. Biến thể

### 13.1. Chỉ pha lên (luôn đi a → z)
Đây chính là `wrong_only_up` ở trên — trở thành đúng nếu đề đổi yêu cầu. Nó tạo các "lớp" chữ phân biệt: `"abcabcabcabc"`.

### 13.2. Gom thành các "lớp" (layers)
Trả về list các chuỗi, mỗi chuỗi là một pha. Hữu ích để kiểm tra hiểu thuật toán.

In [18]:
def layers(s):
    count = Counter(s)
    keys = sorted(count)
    out = []
    up = True
    while sum(count.values()):
        order = keys if up else keys[::-1]
        layer = []
        for ch in order:
            if count[ch]:
                layer.append(ch)
                count[ch] -= 1
        out.append("".join(layer))
        up = not up
    return out

print(layers("aaaabbbbcccc"))
print(layers("leetcode"))

['abc', 'cba', 'abc', 'cba']
['cdelot', 'e', 'e']


### 13.3. Custom order (ý tưởng LeetCode 791)
Thay bảng chữ cái `a..z` bằng một thứ tự tuỳ ý `order`, vẫn đi lên/xuống theo thứ tự đó.

In [19]:
def sort_string_custom(s, order):
    count = Counter(s)
    keys = [ch for ch in order if ch in count]
    res = []
    while len(res) < len(s):
        for ch in keys + keys[::-1]:
            if count[ch]:
                res.append(ch)
                count[ch] -= 1
    return "".join(res)

print(sort_string_custom("aaaabbbbcccc", "cba"))   # đi c → a rồi a → c

cbaabccbaabc


## 14. Bài tập (TODO — đáp án ở cuối)

### Bài 1 — Tính tay
Không chạy code, viết kết quả của `sortString("ggggzzab")`. Lập bảng vòng/pha như mục 6.

### Bài 2 — Đếm tần suất
Viết `exercise_count26(s)` trả về list 26 số đếm.

In [20]:
def exercise_count26(s):
    # TODO
    pass

# Gợi ý test:
# assert exercise_count26("abca")[:3] == [2, 1, 1]

### Bài 3 — Hoàn thiện sortString

In [21]:
def exercise_sort_string(s):
    # TODO 1: đếm
    # TODO 2: while chưa đủ len(s): pha lên a→z, pha xuống z→a
    # TODO 3: ''.join
    pass

# Gợi ý test:
# assert exercise_sort_string("aaaabbbbcccc") == "abccbaabccba"
# assert exercise_sort_string("rat") == "art"

### Bài 4 — Số pha
Trả về **số pha khác rỗng** mà thuật toán thực hiện (mỗi pha lên hoặc xuống lấy ít nhất 1 ký tự). Làm trong `O(n)` mà **không** mô phỏng (gợi ý: số pha phụ thuộc vào tần suất lớn nhất).

In [22]:
def exercise_count_phases(s):
    # TODO
    pass

# Gợi ý test:
# assert exercise_count_phases("aaaabbbbcccc") == 4
# assert exercise_count_phases("rat") == 1
# assert exercise_count_phases("leetcode") == 3

### Bài 5 — Biến thể: bắt đầu bằng pha xuống
Viết `exercise_sort_string_down_first(s)`: mỗi vòng đi **xuống trước**, rồi lên.

In [23]:
def exercise_sort_string_down_first(s):
    # TODO
    pass

# Gợi ý test:
# assert exercise_sort_string_down_first("aaaabbbbcccc") == "cbaabccbaabc"
# assert exercise_sort_string_down_first("rat") == "tra"

### Bài 6 — Mini challenge: kiểm tra một chuỗi có phải output hợp lệ
Cho `s` và `t`, trả `True` nếu `t == sortString(s)` mà **không** gọi lại `sortString`. Ý tưởng: tách `t` thành các pha tăng/giảm nghiêm ngặt và kiểm tra mỗi pha lấy đúng mọi chữ còn hàng.

In [24]:
def exercise_is_valid_output(s, t):
    # TODO
    pass

# Gợi ý test:
# assert exercise_is_valid_output("aaaabbbbcccc", "abccbaabccba") is True
# assert exercise_is_valid_output("aaaabbbbcccc", "abcabcabcabc") is False
# assert exercise_is_valid_output("rat", "art") is True

## 15. Gợi ý 3 mức

**Bài 1** — (1) Đếm `a:1, b:1, g:4, z:2`. (2) Pha lên lấy mỗi chữ còn hàng một lần. (3) Các pha: `abgz`, `zg`, `g`, `g`.

**Bài 2** — (1) Mảng 26 ô. (2) Chỉ số `ord(ch) - 97`. (3) `count[...] += 1`.

**Bài 3** — (1) Xem mục 5. (2) Hai vòng `for` trong `while`. (3) `range(26)` và `range(25, -1, -1)`.

**Bài 4** — (1) Mỗi pha lấy một bản của mỗi chữ còn hàng. (2) Chữ có tần suất `f` lớn nhất còn hàng tới pha thứ `f`. (3) Đáp án = `max(Counter(s).values())`.

**Bài 5** — (1) Đổi thứ tự hai vòng `for`. (2) `order = range(25,-1,-1)` rồi `range(26)`. (3) Giữ điều kiện `while`.

**Bài 6** — (1) Mô phỏng song song với bộ đếm. (2) Duyệt các pha: pha lên cần đúng chuỗi các chữ còn hàng theo `a→z`. (3) So sánh `t[pos:pos+len(layer)]` với `layer` dựng từ bộ đếm, rồi giảm bộ đếm.

## 16. Đáp án tham khảo

**Bài 1:** `abgz` + `zg` + `g` + `g` = `"abgzzggg"`.

In [25]:
assert Solution().sortString("ggggzzab") == "abgzzggg"

def answer_count26(s):
    count = [0] * 26
    for ch in s:
        count[ord(ch) - 97] += 1
    return count

def answer_sort_string(s):
    count = answer_count26(s)
    res = []
    while len(res) < len(s):
        for k in list(range(26)) + list(range(25, -1, -1)):
            if count[k]:
                res.append(chr(k + 97))
                count[k] -= 1
    return "".join(res)

def answer_count_phases(s):
    return max(Counter(s).values())

def answer_sort_string_down_first(s):
    count = answer_count26(s)
    res = []
    while len(res) < len(s):
        for k in list(range(25, -1, -1)) + list(range(26)):
            if count[k]:
                res.append(chr(k + 97))
                count[k] -= 1
    return "".join(res)

def answer_is_valid_output(s, t):
    if len(s) != len(t):
        return False
    count = Counter(s)
    keys = sorted(count)
    pos, up = 0, True
    while pos < len(t):
        layer = "".join(ch for ch in (keys if up else keys[::-1]) if count[ch])
        if t[pos:pos + len(layer)] != layer:
            return False
        for ch in layer:
            count[ch] -= 1
        pos += len(layer)
        up = not up
    return True

assert answer_count26("abca")[:3] == [2, 1, 1]
assert answer_sort_string("aaaabbbbcccc") == "abccbaabccba" and answer_sort_string("rat") == "art"
assert answer_count_phases("aaaabbbbcccc") == 4
assert answer_count_phases("rat") == 1
assert answer_count_phases("leetcode") == 3
assert answer_sort_string_down_first("aaaabbbbcccc") == "cbaabccbaabc"
assert answer_sort_string_down_first("rat") == "tra"
assert answer_is_valid_output("aaaabbbbcccc", "abccbaabccba") is True
assert answer_is_valid_output("aaaabbbbcccc", "abcabcabcabc") is False
assert answer_is_valid_output("rat", "art") is True

rng = random.Random(3)
for _ in range(200):
    t = "".join(rng.choice("abcde") for _ in range(rng.randint(1, 30)))
    assert answer_count_phases(t) == len(layers(t))
    assert answer_is_valid_output(t, sort_string_count(t))
print("Đáp án bài tập: PASS toàn bộ")

Đáp án bài tập: PASS toàn bộ


## 17. Tổng kết — template nhớ nhanh + checklist

```python
count = [0] * 26
for ch in s: count[ord(ch) - 97] += 1
order = list(range(26)) + list(range(25, -1, -1))   # lên rồi xuống
res = []
while len(res) < len(s):
    for k in order:
        if count[k]:
            res.append(chr(k + 97)); count[k] -= 1
return "".join(res)
```

**Checklist trước khi submit**
- [ ] Mỗi pha lấy **một** bản của mỗi chữ còn hàng (không lấy hết).
- [ ] Có cả pha lên **và** pha xuống trong mỗi vòng.
- [ ] Pha xuống duyệt tới `k = 0` (`range(25, -1, -1)`).
- [ ] Điều kiện dừng `len(res) < len(s)` (không lặp vô hạn).
- [ ] Ghép bằng `''.join(res)`; đã test chuỗi 1 ký tự, toàn một chữ, `"rat"`.